### DEAM dataset - Database for Emotional Analysis of Music
[DEAM - Kaggle](https://www.kaggle.com/datasets/imsparsh/deam-mediaeval-dataset-emotional-analysis-in-music)\
[DEAM - Université de Genève](https://cvml.unige.ch/databases/DEAM/)

Prepare data

In [ ]:
# import sys
# import os

# # sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "src")))
# sys.path.append(os.path.abspath(os.path.join("../src")))

# from audio_processing import (
#     load_mean_valence_arousal,
#     load_valence_arousal,
#     get_song_ids_and_labels,
#     DEAMSegmentGenerator,
# )

# print(load_mean_valence_arousal(os.path.abspath(os.path.join("../data/"))))

In [ ]:
# print(load_valence_arousal(os.path.abspath(os.path.join("../data/"))))

In [ ]:
# print(load_valence_arousal(os.path.abspath(os.path.join("../data/")), span=2))

Właściwe przygotowanie danych

In [ ]:
import tensorflow as tf

gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, False)
        tf.config.set_logical_device_configuration(
            gpus[0],
            [
                tf.config.LogicalDeviceConfiguration(memory_limit=4000)
            ],  # Limit to 4000 MB of VRAM (for my 3050 Ti laptop GPU)
        )
    except RuntimeError as e:
        print(e)

In [ ]:
import sys
import os
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath(os.path.join("../src")))
from model import MER_CNN_Model, MER_CNN_Simple, MER_CNN_VGG_Style, MER_CRNN, MER_CNN_MobileNet
from audio_processing import DEAMSegmentGenerator, preprocess_deam, load_valence_arousal, get_song_ids_and_labels

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [ ]:
# PARAMETRY GLOBALNE - MUSZĄ BYĆ SPÓJNE W CAŁYM PROJEKCIE
SR = 22050  # Domyślna częstotliwość librosa.load
HOP_LENGTH = 512  # Krok przesunięcia okna FFT
SEGMENT_WIDTH = 128  # Szerokość obrazka wejściowego do sieci (oś czasu)
N_MELS = 128  # Wysokość obrazka wejściowego do sieci (oś częstotliwości)
SPAN = 3.0  # Jak bardzo uśredniamy etykiety w DataFrame (w sekundach)
DEAM_DIR = os.path.abspath(os.path.join("../data/"))
AUDIO_DIR = os.path.abspath(os.path.join("../data/DEAM/MEMD_audio/"))
AUDIO_NPY_DIR = os.path.abspath(os.path.join("../data/DEAM/audio_npy/"))

In [ ]:
labels_df = load_valence_arousal(DEAM_DIR, span=SPAN)
dynamic_labels = get_song_ids_and_labels(labels_df)

In [ ]:
# Run only once to preprocess audio files into Mel spectrograms and save as .npy
# preprocess_deam(AUDIO_DIR, AUDIO_NPY_DIR, n_mels=N_MELS, hop_length=HOP_LENGTH, sr=SR)

In [ ]:
existing_song_ids = [
    s_id for s_id in dynamic_labels.keys() if os.path.exists(os.path.join(AUDIO_NPY_DIR, f"{s_id}.npy"))
]

print(f"Znaleziono {len(existing_song_ids)} utworów z danymi audio i etykietami.")

In [ ]:
train_ids, val_ids = train_test_split(existing_song_ids, test_size=0.2)
print(f"Trening na {len(train_ids)} utworach, walidacja na {len(val_ids)} utworach.")

In [ ]:
train_generator = DEAMSegmentGenerator(
    song_ids=train_ids,
    labels_dict=dynamic_labels,
    data_dir=AUDIO_NPY_DIR,
    segment_width=SEGMENT_WIDTH,
    hop_size=HOP_LENGTH,
    sr=SR,
    batch_size=32,
    shuffle=True,
)
valid_generator = DEAMSegmentGenerator(
    song_ids=val_ids,
    labels_dict=dynamic_labels,
    data_dir=AUDIO_NPY_DIR,
    segment_width=SEGMENT_WIDTH,
    hop_size=HOP_LENGTH,
    sr=SR,
    batch_size=32,
    shuffle=False,
)

In [ ]:
model_CNN = MER_CNN_Simple()
model_CNN.fit(
    train_generator,
    validation_generator=valid_generator,
    epochs=50,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
        ModelCheckpoint("best_model.h5", save_best_only=True),
    ],
)